# Extract Enformer embeddings from scATAC-seq peaks

In [1]:
import os
import sys
sys.path.insert(0, "/staging/leuven/stg_00002/lcb/gpartel/software/deepSCENIC/")
from src.utils import format_region_to_bed

import torch
from torch.utils.data import DataLoader
from enformer_pytorch import Enformer, GenomeIntervalDataset

import numpy as np
import scanpy as sc
from tqdm import tqdm
import h5py

urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2023-10-01 00:23:27.918603: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-01 00:23:38.116721: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-10-01 00:23:42.546881: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory f

In [2]:
DATA_PATH = '/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/mouse_evo/'

seq_len = 640
regions = sc.read(DATA_PATH + 'fragment_matrix.h5ad').var_names
regs = []
for r in regions:
    chr, coords = r.split(':')
    start, end = np.array(coords.split('-')).astype(int)
    if end-start > seq_len:
        center = (end-start)//2
        start = center - seq_len//2
        end = center + seq_len//2
        regs.append(chr + ':' + str(start) + '-' + str(end))
    else:
        regs.append(r)

In [3]:
output_file = open(DATA_PATH + 'regions.bed', 'w')
[format_region_to_bed(r, output_file) for r in regs]
output_file.close()

In [10]:
ds = GenomeIntervalDataset(
    bed_file = DATA_PATH + 'regions.bed',                       # bed file - columns 0, 1, 2 must be <chromosome>, <start position>, <end position>
    fasta_file = '/lustre1/project/stg_00002/lcb/gpartel/resources/genomes/mm10.fa', # path to fasta file
    return_seq_indices = False,                          # return nucleotide indices (ACGTN) or one hot encodings
    shift_augs = (-0, 0),                               # random shift augmentations from -2 to +2 basepairs
    rc_aug = False,                                      # use reverse complement augmentation with 50% probability
    context_length = seq_len,
    return_augs = False                                  # return the augmentation meta data
)

In [11]:
model = Enformer.from_pretrained('EleutherAI/enformer-official-rough', target_length=5, dropout_rate = 0).to('cuda')

In [12]:
dataloader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)

In [13]:
h5_name = DATA_PATH + '/enformer_embs.h5'
print(not os.path.exists(h5_name))
if not os.path.exists(h5_name):
    f = h5py.File(h5_name, "w")

    ds_X = f.create_dataset(
        "X",
        (len(regions), 5, 3072), # 5 is the number of tokens, 3072 is the embedding size
        dtype="float32",
    )

True


In [14]:
Tensor = torch.cuda.FloatTensor
    
model.eval()
embs = []
i = 0
with torch.no_grad():
    for j, data_batch in tqdm(enumerate(dataloader, 0), unit="batch", total=len(dataloader)):
        data_batch = data_batch.to('cuda')
        _, emb = model(data_batch, return_embeddings = True)
        ds_X[i:i+emb.shape[0]] = emb.cpu().numpy().astype('float32')
        i = i+emb.shape[0]

f.close()

100%|██████████| 7030/7030 [07:50<00:00, 14.94batch/s]


In [15]:
len(regions)

449886